In [20]:
import pandas as pd
data=pd.read_csv('HousePricePrediction.csv')

In [21]:
data.head(20)

,Id,MSSubClass,MSZoning,LotArea,LotConfig,BldgType,OverallCond,YearBuilt,YearRemodAdd,Exterior1st,BsmtFinSF2,TotalBsmtSF,SalePrice
0,0,60,RL,8450,Inside,1Fam,5,2003,2003,VinylSd,0.0,856.0,208500.0
1,1,20,RL,9600,FR2,1Fam,8,1976,1976,MetalSd,0.0,1262.0,181500.0
2,2,60,RL,11250,Inside,1Fam,5,2001,2002,VinylSd,0.0,920.0,223500.0
3,3,70,RL,9550,Corner,1Fam,5,1915,1970,Wd Sdng,0.0,756.0,140000.0
4,4,60,RL,14260,FR2,1Fam,5,2000,2000,VinylSd,0.0,1145.0,250000.0
5,5,50,RL,14115,Inside,1Fam,5,1993,1995,VinylSd,0.0,796.0,143000.0
6,6,20,RL,10084,Inside,1Fam,5,2004,2005,VinylSd,0.0,1686.0,307000.0
7,7,60,RL,10382,Corner,1Fam,6,1973,1973,HdBoard,32.0,1107.0,200000.0
8,8,50,RM,6120,Inside,1Fam,5,1931,1950,BrkFace,0.0,952.0,129900.0
9,9,190,RL,7420,Corner,2fmCon,6,1939,1950,MetalSd,0.0,991.0,118000.0


In [24]:
# data['BldgType'].unique()
data['MSZoning'].unique()
# data['LotConfig'].unique()
# data['Exterior1st'].unique()

array(['RL', 'RM', 'C (all)', 'FV', 'RH', nan], dtype=object)

In [25]:
data.info()

# Important rule (very important for ML)

# Never fill SalePrice for test data arbitrarily

# Correct workflow:

# Train model using rows where SalePrice is NOT null

# Predict SalePrice for rows where it IS null

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2919 entries, 0 to 2918
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Id            2919 non-null   int64  
 1   MSSubClass    2919 non-null   int64  
 2   MSZoning      2915 non-null   object 
 3   LotArea       2919 non-null   int64  
 4   LotConfig     2919 non-null   object 
 5   BldgType      2919 non-null   object 
 6   OverallCond   2919 non-null   int64  
 7   YearBuilt     2919 non-null   int64  
 8   YearRemodAdd  2919 non-null   int64  
 9   Exterior1st   2918 non-null   object 
 10  BsmtFinSF2    2918 non-null   float64
 11  TotalBsmtSF   2918 non-null   float64
 12  SalePrice     1460 non-null   float64
dtypes: float64(3), int64(6), object(4)
memory usage: 296.6+ KB


In [5]:
data.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotArea', 'LotConfig', 'BldgType',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'Exterior1st', 'BsmtFinSF2',
       'TotalBsmtSF', 'SalePrice'],
      dtype='object')

In [26]:
data['BsmtFinSF2']=data['BsmtFinSF2'].fillna(data['BsmtFinSF2'].median())
data['TotalBsmtSF']=data['TotalBsmtSF'].fillna(data['TotalBsmtSF'].median())

In [27]:
data['MSZoning']=data['MSZoning'].fillna(data['MSZoning'].mode()[0])  #mode gives series so choose one
data['Exterior1st']=data['Exterior1st'].fillna(data['Exterior1st'].mode()[0])

In [28]:
data['MSZoning']=data['MSZoning'].map({'RL':1, 'RM':2, 'C (all)':3, 'FV':4, 'RH':5,})
data['LotConfig']=data['LotConfig'].map({'Inside':1, 'FR2':2, 'Corner':3, 'CulDSac':4, 'FR3':5})
data['BldgType']=data['BldgType'].map({'1Fam':1, '2fmCon':2, 'Duplex':3, 'TwnhsE':4, 'Twnhs':5})
data['Exterior1st']=data['Exterior1st'].map({'VinylSd':1, 'MetalSd':2, 'Wd Sdng':3, 'HdBoard':4, 'BrkFace':5, 'WdShing':6,
       'CemntBd':7, 'Plywood':8, 'AsbShng':9, 'Stucco':10, 'BrkComm':11, 'AsphShn':12,
       'Stone':13, 'ImStucc':14, 'CBlock':15})

In [29]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2919 entries, 0 to 2918
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Id            2919 non-null   int64  
 1   MSSubClass    2919 non-null   int64  
 2   MSZoning      2919 non-null   int64  
 3   LotArea       2919 non-null   int64  
 4   LotConfig     2919 non-null   int64  
 5   BldgType      2919 non-null   int64  
 6   OverallCond   2919 non-null   int64  
 7   YearBuilt     2919 non-null   int64  
 8   YearRemodAdd  2919 non-null   int64  
 9   Exterior1st   2919 non-null   int64  
 10  BsmtFinSF2    2919 non-null   float64
 11  TotalBsmtSF   2919 non-null   float64
 12  SalePrice     1460 non-null   float64
dtypes: float64(3), int64(10)
memory usage: 296.6 KB


In [31]:
known_data=data[data['SalePrice'].notnull()]  #important
x_train=known_data.drop(columns=['Id','SalePrice'])
y_train=known_data['SalePrice']


from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    x_train, y_train, test_size=0.2, random_state=42)

In [32]:
X_test.head()

,MSSubClass,MSZoning,LotArea,LotConfig,BldgType,OverallCond,YearBuilt,YearRemodAdd,Exterior1st,BsmtFinSF2,TotalBsmtSF
892,20,1,8414,1,1,8,1963,2003,4,0.0,1059.0
1105,60,1,12256,3,1,5,1994,1995,4,0.0,1463.0
413,30,2,8960,1,1,6,1927,1950,6,0.0,1008.0
522,50,2,5000,3,1,7,1947,1950,7,0.0,1004.0
1036,20,1,12898,1,1,5,2007,2008,1,0.0,1620.0


In [38]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [39]:
X_test.shape
y_pred=model.predict(X_test)

In [41]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)
print("r-squared:", r2)
n = X_test.shape[0]
p = X_test.shape[1]

adjusted_r2 = 1 - ((1-r2) * (n-1) / (n-p-1))
print("adjusted r^2:", adjusted_r2)

r-squared: 0.6180875402374884
adjusted r^2: 0.603083836461104


In [42]:
unknown_data_with_pred = unknown_data.copy()


In [43]:
Y_predict

array([134637.75121378, 169064.92010216, 211622.62666141, ...,
       197132.05872696, 215461.40854158, 210604.13678673], shape=(1459,))

In [45]:
unknown_data_with_pred = unknown_data.copy()
unknown_data_with_pred['SalePrice_Predicted'] = Y_predict
unknown_data_with_pred

,Id,MSSubClass,MSZoning,LotArea,LotConfig,BldgType,OverallCond,YearBuilt,YearRemodAdd,Exterior1st,BsmtFinSF2,TotalBsmtSF,SalePrice,SalePrice_Predicted
1460,1460,20,5,11622,1,1,6,1961,1961,1,144.0,882.0,NaN,134637.751214
1461,1461,20,1,14267,3,1,6,1958,1958,3,0.0,1329.0,NaN,169064.920102
1462,1462,60,1,13830,1,1,5,1997,1998,1,0.0,928.0,NaN,211622.626661
1463,1463,60,1,9978,1,1,6,1998,1998,1,0.0,926.0,NaN,214277.831295
1464,1464,120,1,5005,1,4,5,1992,1992,4,0.0,1280.0,NaN,181012.130870
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2914,2914,160,2,1936,1,5,7,1970,1970,7,0.0,546.0,NaN,95245.294748
2915,2915,160,2,1894,1,4,5,1970,1970,7,0.0,546.0,NaN,111313.983012
2916,2916,20,1,20000,1,1,7,1960,1996,1,0.0,1224.0,NaN,197132.058727
2917,2917,85,1,10441,1,1,5,1992,1992,4,0.0,912.0,NaN,215461.408542
